# Classical Baseline Models

This notebook trains leakage-safe classical baselines for overload-risk classification and line-flow forecasting. Hyperparameter choices and calibration use training/validation partitions only, while test and external-test partitions are held out for final comparison.

In [1]:
import os, json, math, time, zipfile, subprocess, warnings, textwrap, hashlib
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import display, Image
warnings.filterwarnings("ignore")

mpl.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 17,
    "axes.titlesize": 18,
    "axes.labelsize": 17,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 16,
    "figure.titlesize": 18,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "savefig.dpi": 350,
    "figure.dpi": 350,
})

PALETTE = {
    "blue": "#2F6B8F",
    "orange": "#C77C2B",
    "green": "#4F7F3A",
    "purple": "#6D5A8D",
    "red": "#B45A55",
    "gray": "#6E7378",
    "light_gray": "#E7EAED",
    "teal": "#4A8C8A",
    "gold": "#C9A227",
    "black": "#222222",
    "white": "#FFFFFF",
}

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "Notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_RAW = PROJECT_ROOT / "Data" / "Raw"
DATA_UNIFIED = PROJECT_ROOT / "Data" / "Unified"
DATA_EXTERNAL = PROJECT_ROOT / "Data" / "External_Test"
TABLES = PROJECT_ROOT / "Tables"
FIGURES = PROJECT_ROOT / "Figures"
MODELS = PROJECT_ROOT / "Models"
REPORTS = PROJECT_ROOT / "Reports"
for p in [DATA_RAW, DATA_UNIFIED, DATA_EXTERNAL, TABLES, FIGURES, MODELS, REPORTS, PROJECT_ROOT / "Drawio_Preview"]:
    p.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Random seed: {RANDOM_SEED}")

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score, brier_score_loss, confusion_matrix, mean_absolute_error,
    mean_squared_error, r2_score)
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression, BayesianRidge, Ridge, ElasticNet
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPClassifier
from sklearn.inspection import permutation_importance
import joblib

Project root: /Users/talgatazykanov/Desktop/Science works/Bayiessian_Asem_Almaty_2026_05/Bayesian_PowerGrid_Risk_Forecasting_N1
Random seed: 42


## Load Modelling Dataset and Define Leakage-Safe Features

Future targets, risk labels, split identifiers, timestamps, and row identifiers are excluded from the feature matrix. The primary comparison uses the 1-hour Critical-risk target and the 1-hour line-flow target.

In [2]:
model_df = pd.read_parquet(DATA_UNIFIED / "unified_powergrid_modeling_dataset.parquet")
feature_dict = pd.read_csv(DATA_UNIFIED / "feature_dictionary.csv")
feature_cols = feature_dict[(feature_dict["role"] == "feature") & (feature_dict["dtype"].str.contains("float|int", case=False, regex=True))]["column"].tolist()
feature_cols = [c for c in feature_cols if c in model_df.columns and c not in ["year", "time_index"]]
coordinate_cols_forbidden = {"from_longitude", "from_latitude", "to_longitude", "to_latitude"}
leaking_coordinate_features = sorted(coordinate_cols_forbidden.intersection(feature_cols))
assert not leaking_coordinate_features, f"Raw coordinate columns must remain metadata only: {leaking_coordinate_features}"
# Keep time_index as a feature only through cyclic/calendar columns; raw time index is retained for diagnostics.
print("Rows:", len(model_df), "Feature columns:", len(feature_cols))
display(feature_dict[feature_dict["role"] == "feature"].groupby("feature_group").size().reset_index(name="columns"))

Rows: 3484800 Feature columns: 69


,feature_group,columns
0,generation,14
1,graph,4
2,line_flow_history,23
3,load,10
4,operating_regime,3
5,other_numeric,3
6,temporal,10
7,topology,4


In [3]:
def sample_by_split(df, split, n, stratify_col="risk_binary_h1", seed=RANDOM_SEED):
    sub = df[df["split"] == split]
    if len(sub) <= n:
        return sub.copy()
    parts = []
    for val, grp in sub.groupby(stratify_col):
        take = max(1, int(n * len(grp) / len(sub)))
        parts.append(grp.sample(min(take, len(grp)), random_state=seed))
    out = pd.concat(parts).sample(frac=1, random_state=seed)
    if len(out) > n:
        out = out.sample(n, random_state=seed)
    return out.copy()

def balanced_train_sample(df, n=120000, seed=RANDOM_SEED):
    tr = df[df["split"] == "train"]
    pos = tr[tr["risk_binary_h1"] == 1]
    neg = tr[tr["risk_binary_h1"] == 0]
    n_pos = min(len(pos), max(int(n * 0.35), 1000))
    n_neg = min(len(neg), n - n_pos)
    out = pd.concat([pos.sample(n_pos, random_state=seed), neg.sample(n_neg, random_state=seed)]).sample(frac=1, random_state=seed)
    return out.copy()

train_df = balanced_train_sample(model_df, n=120000)
val_df = sample_by_split(model_df, "validation", 60000)
test_df = sample_by_split(model_df, "test", 60000)
ext_df = sample_by_split(model_df, "external_test", 60000)
for name, sub in [("train", train_df), ("validation", val_df), ("test", test_df), ("external_test", ext_df)]:
    print(name, sub.shape, "critical rate", float(sub["risk_binary_h1"].mean()))

train (120000, 94) critical rate 0.3499999940395355
validation (59999, 94) critical rate 0.04361739382147789
test (59999, 94) critical rate 0.07898464798927307
external_test (59999, 94) critical rate 0.05426757037639618


## Metric Functions

Classification metrics include threshold-dependent scores, ranking metrics, Brier score, and expected calibration error. Forecasting metrics include scale-dependent, normalized, ramp-sensitive, and probabilistic interval scores.

In [4]:
def expected_calibration_error(y_true, prob, n_bins=10):
    y_true = np.asarray(y_true).astype(float)
    prob = np.asarray(prob).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (prob >= lo) & (prob < hi if hi < 1 else prob <= hi)
        if mask.any():
            ece += mask.mean() * abs(y_true[mask].mean() - prob[mask].mean())
    return float(ece)

def classification_metrics(y_true, prob, pred=None):
    y_true = np.asarray(y_true).astype(int)
    prob = np.nan_to_num(np.asarray(prob).astype(float), nan=0.0, posinf=1.0, neginf=0.0)
    prob = np.clip(prob, 0, 1)
    if pred is None:
        pred = (prob >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0,1]).ravel()
    return {
        "Accuracy": accuracy_score(y_true, pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, pred),
        "Macro-F1": f1_score(y_true, pred, average="macro", zero_division=0),
        "Weighted-F1": f1_score(y_true, pred, average="weighted", zero_division=0),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "Specificity": tn / (tn + fp + 1e-12),
        "ROC-AUC": roc_auc_score(y_true, prob) if len(np.unique(y_true)) > 1 else np.nan,
        "PR-AUC": average_precision_score(y_true, prob) if len(np.unique(y_true)) > 1 else np.nan,
        "False Positive Rate": fp / (fp + tn + 1e-12),
        "False Negative Rate": fn / (fn + tp + 1e-12),
        "Brier Score": brier_score_loss(y_true, prob),
        "Expected Calibration Error": expected_calibration_error(y_true, prob),
    }

def smape(y, yhat):
    y, yhat = np.asarray(y), np.asarray(yhat)
    return float(np.mean(2 * np.abs(yhat - y) / (np.abs(y) + np.abs(yhat) + 1e-6)))

def forecasting_metrics(y_true, pred, current_flow=None):
    y_true = np.asarray(y_true).astype(float)
    pred = np.asarray(pred).astype(float)
    rmse = mean_squared_error(y_true, pred) ** 0.5
    denom = np.maximum(np.abs(y_true), 1e-3)
    ramp_mae = np.nan
    if current_flow is not None:
        ramp_mae = mean_absolute_error(y_true - np.asarray(current_flow), pred - np.asarray(current_flow))
    return {
        "MAE": mean_absolute_error(y_true, pred),
        "RMSE": rmse,
        "sMAPE": smape(y_true, pred),
        "MAPE_safe": float(np.mean(np.abs((y_true - pred) / denom))),
        "R2": r2_score(y_true, pred),
        "normalized_RMSE": rmse / (np.nanstd(y_true) + 1e-9),
        "ramp_MAE": ramp_mae,
    }

def interval_metrics(y, lo, hi):
    y = np.asarray(y); lo = np.asarray(lo); hi = np.asarray(hi)
    return {
        "interval_coverage_probability": float(((y >= lo) & (y <= hi)).mean()),
        "mean_prediction_interval_width": float(np.mean(hi - lo)),
        "pinball_loss_q05": float(np.mean(np.maximum(0.05*(y-lo), (0.05-1)*(y-lo)))),
        "pinball_loss_q95": float(np.mean(np.maximum(0.95*(y-hi), (0.95-1)*(y-hi)))),
        "interval_calibration_error": float(abs(((y >= lo) & (y <= hi)).mean() - 0.90)),
    }
print("Metric functions ready.")

Metric functions ready.


## Risk Classification Baselines

Persistence and previous-day baselines are evaluated alongside probabilistic classical models. Optional XGBoost, LightGBM, and CatBoost are used when available and otherwise recorded as unavailable.

In [5]:
(MODELS / "baseline_model_artifacts").mkdir(parents=True, exist_ok=True)
X_train, y_train = train_df[feature_cols].replace([np.inf, -np.inf], np.nan), train_df["risk_binary_h1"].astype(int)
X_val, y_val = val_df[feature_cols].replace([np.inf, -np.inf], np.nan), val_df["risk_binary_h1"].astype(int)
X_test, y_test = test_df[feature_cols].replace([np.inf, -np.inf], np.nan), test_df["risk_binary_h1"].astype(int)
X_ext, y_ext = ext_df[feature_cols].replace([np.inf, -np.inf], np.nan), ext_df["risk_binary_h1"].astype(int)

pre = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
plain_pre = Pipeline([("imputer", SimpleImputer(strategy="median"))])

classifiers = {
    "Gaussian Naive Bayes": Pipeline([("prep", pre), ("model", GaussianNB())]),
    "Logistic Regression": Pipeline([("prep", pre), ("model", LogisticRegression(max_iter=600, class_weight="balanced", n_jobs=-1, random_state=RANDOM_SEED))]),
    "Random Forest": Pipeline([("prep", plain_pre), ("model", RandomForestClassifier(n_estimators=120, max_depth=14, min_samples_leaf=8, class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_SEED))]),
    "Extra Trees": Pipeline([("prep", plain_pre), ("model", ExtraTreesClassifier(n_estimators=160, max_depth=16, min_samples_leaf=5, class_weight="balanced", n_jobs=-1, random_state=RANDOM_SEED))]),
    "HistGradientBoostingClassifier": Pipeline([("prep", plain_pre), ("model", HistGradientBoostingClassifier(max_iter=160, learning_rate=0.06, max_leaf_nodes=31, l2_regularization=0.05, random_state=RANDOM_SEED))]),
    "MLPClassifier": Pipeline([("prep", pre), ("model", MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=80, early_stopping=True, random_state=RANDOM_SEED))]),
}
optional_status = []
try:
    from xgboost import XGBClassifier
    classifiers["XGBoost"] = Pipeline([("prep", plain_pre), ("model", XGBClassifier(n_estimators=120, max_depth=4, learning_rate=0.07, subsample=0.85, colsample_bytree=0.85, eval_metric="logloss", tree_method="hist", random_state=RANDOM_SEED))])
    optional_status.append({"model":"XGBoost", "status":"available"})
except Exception as e:
    optional_status.append({"model":"XGBoost", "status":f"unavailable: {e}"})
try:
    from lightgbm import LGBMClassifier
    classifiers["LightGBM"] = Pipeline([("prep", plain_pre), ("model", LGBMClassifier(n_estimators=160, learning_rate=0.06, max_depth=-1, num_leaves=31, class_weight="balanced", random_state=RANDOM_SEED, verbose=-1))])
    optional_status.append({"model":"LightGBM", "status":"available"})
except Exception as e:
    optional_status.append({"model":"LightGBM", "status":f"unavailable: {e}"})
try:
    import catboost  # noqa: F401
    optional_status.append({"model":"CatBoost", "status":"available but skipped: incompatible sklearn tag API in this environment"})
except Exception as e:
    optional_status.append({"model":"CatBoost", "status":f"unavailable: {e}"})

class_rows = []
prediction_rows = []
# Rule baselines.
rule_specs = {
    "Persistence risk baseline": lambda d: (d["risk_proximity_score"].to_numpy() >= 1.0).astype(float),
    "Seasonal previous-day risk baseline": lambda d: (d["absolute_line_flow_lag24"].to_numpy() >= d["abs_flow_q95_train"].to_numpy()).astype(float),
}
for model_name, fn in rule_specs.items():
    for split, sub, y in [("validation", val_df, y_val), ("test", test_df, y_test), ("external_test", ext_df, y_ext)]:
        t_pred = time.time()
        prob = fn(sub)
        inference_time = time.time() - t_pred
        met = classification_metrics(y, prob)
        class_rows.append({"model": model_name, "split": split, "inference_time_seconds": inference_time, **met})
        prediction_rows.append(pd.DataFrame({"row_id": sub["row_id"], "split": split, "model": model_name, "task": "classification", "y_true": y, "y_prob": prob, "y_pred": (prob >= 0.5).astype(int)}))

# Bayesian ridge approximation.
br_prep = pre.fit(X_train)
Xtr_s = br_prep.transform(X_train)
br = BayesianRidge().fit(Xtr_s, y_train)
for split, X, sub, y in [("validation", X_val, val_df, y_val), ("test", X_test, test_df, y_test), ("external_test", X_ext, ext_df, y_ext)]:
    t_pred = time.time()
    mean, std = br.predict(br_prep.transform(X), return_std=True)
    inference_time = time.time() - t_pred
    prob = np.clip(mean, 0, 1)
    met = classification_metrics(y, prob)
    class_rows.append({"model": "Bayesian Ridge risk approximation", "split": split, "inference_time_seconds": inference_time, **met})
    prediction_rows.append(pd.DataFrame({"row_id": sub["row_id"], "split": split, "model": "Bayesian Ridge risk approximation", "task": "classification", "y_true": y, "y_prob": prob, "y_pred": (prob >= 0.5).astype(int), "posterior_std": std}))

for model_name, clf in classifiers.items():
    t0 = time.time()
    clf.fit(X_train, y_train)
    fit_time = time.time() - t0
    joblib.dump(clf, MODELS / "baseline_model_artifacts" / f"{model_name.replace(' ', '_').replace('/', '_')}_classifier.joblib") if not model_name.startswith("CatBoost") else None
    for split, X, sub, y in [("validation", X_val, val_df, y_val), ("test", X_test, test_df, y_test), ("external_test", X_ext, ext_df, y_ext)]:
        t_pred = time.time()
        if hasattr(clf, "predict_proba"):
            prob = clf.predict_proba(X)[:, 1]
        else:
            scores = clf.decision_function(X)
            prob = 1/(1+np.exp(-scores))
        inference_time = time.time() - t_pred
        pred = (prob >= 0.5).astype(int)
        met = classification_metrics(y, prob, pred)
        class_rows.append({"model": model_name, "split": split, "fit_time_seconds": fit_time, "inference_time_seconds": inference_time, **met})
        prediction_rows.append(pd.DataFrame({"row_id": sub["row_id"], "split": split, "model": model_name, "task": "classification", "y_true": y, "y_prob": prob, "y_pred": pred}))
    print("Trained", model_name, "in", round(fit_time, 2), "s")

class_metrics = pd.DataFrame(class_rows)
class_metrics.to_csv(TABLES / "baseline_classification_metrics.csv", index=False)
display(pd.DataFrame(optional_status))
display(class_metrics.sort_values(["split", "Macro-F1"], ascending=[True, False]).head(20))
print("Saved:", TABLES / "baseline_classification_metrics.csv")

Trained Gaussian Naive Bayes in 1.28 s


Trained Logistic Regression in 3.6 s


Trained Random Forest in 14.42 s


Trained Extra Trees in 4.86 s


Trained HistGradientBoostingClassifier in 17.8 s


...

Trained XGBoost in 4.75 s


Trained LightGBM in 5.62 s


,model,status
0,XGBoost,available
1,LightGBM,available
2,CatBoost,available but skipped: incompatible sklearn ta...


,model,split,inference_time_seconds,Accuracy,Balanced Accuracy,Macro-F1,Weighted-F1,Precision,Recall,Specificity,ROC-AUC,PR-AUC,False Positive Rate,False Negative Rate,Brier Score,Expected Calibration Error,fit_time_seconds
23,HistGradientBoostingClassifier,external_test,0.217523,0.981516,0.979951,0.920931,0.982632,0.754203,0.978194,0.981707,0.997778,0.962938,0.018293,0.021806,0.013847,0.019211,17.795678
32,LightGBM,external_test,0.109725,0.978233,0.981978,0.909686,0.979828,0.718023,0.986179,0.977777,0.997760,0.961790,0.022223,0.013821,0.016705,0.024059,5.617686
29,XGBoost,external_test,0.062191,0.977716,0.977507,0.907240,0.979319,0.715861,0.977273,0.977742,0.997081,0.952451,0.022258,0.022727,0.016825,0.025944,4.750656
2,Persistence risk baseline,external_test,0.000156,0.980216,0.901387,0.903196,0.980172,0.820775,0.812961,0.989814,0.901387,0.677408,0.010186,0.187039,0.019784,0.019784,NaN
26,MLPClassifier,external_test,0.071865,0.977233,0.928181,0.897101,0.978050,0.748946,0.873157,0.983205,0.993232,0.908714,0.016795,0.126843,0.016995,0.013215,24.604572
17,Random Forest,external_test,0.226256,0.972450,0.978197,0.890136,0.974911,0.666667,0.984644,0.971750,0.996652,0.940944,0.028250,0.015356,0.020264,0.035080,14.418811
14,Logistic Regression,external_test,0.040095,0.966483,0.921629,0.860208,0.968866,0.640551,0.871314,0.971944,0.989936,0.861892,0.028056,0.128686,0.024068,0.023726,3.600326
20,Extra Trees,external_test,0.189881,0.959449,0.971324,0.851520,0.964372,0.573627,0.984644,0.958004,0.994870,0.917202,0.041996,0.015356,0.029452,0.054841,4.860805
5,Seasonal previous-day risk baseline,external_test,0.000207,0.957249,0.791538,0.791664,0.957240,0.606210,0.605651,0.977425,0.791538,0.388552,0.022575,0.394349,0.042751,0.042751,NaN
8,Bayesian Ridge risk approximation,external_test,0.053315,0.911015,0.825429,0.711117,0.925342,0.347578,0.729423,0.921435,0.953280,0.616832,0.078565,0.270577,0.055680,0.091803,NaN


Saved: /Users/talgatazykanov/Desktop/Science works/Bayiessian_Asem_Almaty_2026_05/Bayesian_PowerGrid_Risk_Forecasting_N1/Tables/baseline_classification_metrics.csv


## Line-Flow Forecasting Baselines

Forecasting baselines include persistence, previous-day persistence, linear regularized models, tree ensembles, histogram gradient boosting, optional XGBoost, and quantile/conformal intervals.

In [6]:
(MODELS / "baseline_model_artifacts").mkdir(parents=True, exist_ok=True)
ytr_reg = train_df["future_line_flow_h1"].astype(float)
yval_reg = val_df["future_line_flow_h1"].astype(float)
ytest_reg = test_df["future_line_flow_h1"].astype(float)
yext_reg = ext_df["future_line_flow_h1"].astype(float)

regressors = {
    "Ridge": Pipeline([("prep", pre), ("model", Ridge(alpha=3.0, random_state=RANDOM_SEED))]),
    "ElasticNet": Pipeline([("prep", pre), ("model", ElasticNet(alpha=0.002, l1_ratio=0.2, max_iter=2000, random_state=RANDOM_SEED))]),
    "RandomForestRegressor": Pipeline([("prep", plain_pre), ("model", RandomForestRegressor(n_estimators=100, max_depth=16, min_samples_leaf=6, n_jobs=-1, random_state=RANDOM_SEED))]),
    "ExtraTreesRegressor": Pipeline([("prep", plain_pre), ("model", ExtraTreesRegressor(n_estimators=120, max_depth=18, min_samples_leaf=4, n_jobs=-1, random_state=RANDOM_SEED))]),
    "HistGradientBoostingRegressor": Pipeline([("prep", plain_pre), ("model", HistGradientBoostingRegressor(max_iter=180, learning_rate=0.06, max_leaf_nodes=31, l2_regularization=0.05, random_state=RANDOM_SEED))]),
}
try:
    from xgboost import XGBRegressor
    regressors["XGBoost Regressor"] = Pipeline([("prep", plain_pre), ("model", XGBRegressor(n_estimators=140, max_depth=4, learning_rate=0.06, subsample=0.85, colsample_bytree=0.85, objective="reg:squarederror", tree_method="hist", random_state=RANDOM_SEED))])
except Exception as e:
    print("XGBoost Regressor unavailable:", e)

forecast_rows = []
forecast_pred_rows = []
rule_reg = {
    "Persistence forecast": lambda d: d["line_flow"].to_numpy(dtype=float),
    "Seasonal persistence forecast": lambda d: d["line_flow_lag24"].to_numpy(dtype=float),
}
for model_name, fn in rule_reg.items():
    for split, sub, y in [("validation", val_df, yval_reg), ("test", test_df, ytest_reg), ("external_test", ext_df, yext_reg)]:
        t_pred = time.time()
        pred = fn(sub)
        inference_time = time.time() - t_pred
        met = forecasting_metrics(y, pred, sub["line_flow"])
        forecast_rows.append({"model": model_name, "split": split, "inference_time_seconds": inference_time, **met})
        forecast_pred_rows.append(pd.DataFrame({"row_id": sub["row_id"], "split": split, "model": model_name, "task": "forecasting", "y_true": y, "y_pred": pred}))

for model_name, reg in regressors.items():
    t0 = time.time()
    reg.fit(X_train, ytr_reg)
    fit_time = time.time() - t0
    if model_name not in ["RandomForestRegressor", "ExtraTreesRegressor"]:
        joblib.dump(reg, MODELS / "baseline_model_artifacts" / f"{model_name.replace(' ', '_')}_regressor.joblib")
    for split, X, sub, y in [("validation", X_val, val_df, yval_reg), ("test", X_test, test_df, ytest_reg), ("external_test", X_ext, ext_df, yext_reg)]:
        t_pred = time.time()
        pred = reg.predict(X)
        inference_time = time.time() - t_pred
        met = forecasting_metrics(y, pred, sub["line_flow"])
        forecast_rows.append({"model": model_name, "split": split, "fit_time_seconds": fit_time, "inference_time_seconds": inference_time, **met})
        forecast_pred_rows.append(pd.DataFrame({"row_id": sub["row_id"], "split": split, "model": model_name, "task": "forecasting", "y_true": y, "y_pred": pred}))
    print("Trained", model_name, "in", round(fit_time, 2), "s")

# Quantile gradient boosting and conformal interval baseline.
q_models = {}
for q in [0.05, 0.50, 0.95]:
    qm = Pipeline([("prep", plain_pre), ("model", HistGradientBoostingRegressor(loss="quantile", quantile=q, max_iter=120, learning_rate=0.06, max_leaf_nodes=31, random_state=RANDOM_SEED))])
    qm.fit(X_train, ytr_reg)
    q_models[q] = qm
val_med = q_models[0.50].predict(X_val)
val_abs_res = np.abs(yval_reg - val_med)
conformal_q = float(np.quantile(val_abs_res, 0.90))
for split, X, sub, y in [("validation", X_val, val_df, yval_reg), ("test", X_test, test_df, ytest_reg), ("external_test", X_ext, ext_df, yext_reg)]:
    t_pred = time.time()
    lo = q_models[0.05].predict(X) - conformal_q
    med = q_models[0.50].predict(X)
    hi = q_models[0.95].predict(X) + conformal_q
    inference_time = time.time() - t_pred
    met = {**forecasting_metrics(y, med, sub["line_flow"]), **interval_metrics(y, lo, hi)}
    forecast_rows.append({"model": "Quantile Gradient Boosting conformal", "split": split, "fit_time_seconds": np.nan, "inference_time_seconds": inference_time, **met})
    forecast_pred_rows.append(pd.DataFrame({"row_id": sub["row_id"], "split": split, "model": "Quantile Gradient Boosting conformal", "task": "forecasting", "y_true": y, "y_pred": med, "lower_q05": lo, "upper_q95": hi}))

forecast_metrics = pd.DataFrame(forecast_rows)
forecast_metrics.to_csv(TABLES / "baseline_forecasting_metrics.csv", index=False)

preds = pd.concat(prediction_rows + forecast_pred_rows, ignore_index=True)
preds[preds["split"] == "validation"].to_csv(TABLES / "baseline_validation_predictions.csv", index=False)
preds[preds["split"] == "test"].to_csv(TABLES / "baseline_test_predictions.csv", index=False)
preds[preds["split"] == "external_test"].to_csv(TABLES / "baseline_external_test_predictions.csv", index=False)

display(forecast_metrics.sort_values(["split", "MAE"]).head(20))
print("Saved baseline predictions and forecasting metrics.")

Trained Ridge in 1.0 s


Trained ElasticNet in 10.72 s


Trained RandomForestRegressor in 84.27 s


Trained ExtraTreesRegressor in 31.15 s


Trained HistGradientBoostingRegressor in 12.38 s


Trained XGBoost Regressor in 2.74 s


,model,split,inference_time_seconds,MAE,RMSE,sMAPE,MAPE_safe,R2,normalized_RMSE,ramp_MAE,fit_time_seconds,interval_coverage_probability,mean_prediction_interval_width,pinball_loss_q05,pinball_loss_q95,interval_calibration_error
17,ExtraTreesRegressor,external_test,0.493523,0.374802,0.590289,0.112208,0.217639,0.995639,0.066035,0.374802,31.145975,NaN,NaN,NaN,NaN,NaN
20,HistGradientBoostingRegressor,external_test,0.230624,0.384645,0.565258,0.115691,0.224767,0.996001,0.063235,0.384645,12.378600,NaN,NaN,NaN,NaN,NaN
14,RandomForestRegressor,external_test,0.281501,0.385496,0.601816,0.114127,0.212530,0.995467,0.067325,0.385496,84.273529,NaN,NaN,NaN,NaN,NaN
23,XGBoost Regressor,external_test,0.059375,0.431473,0.643516,0.127181,0.249046,0.994818,0.071989,0.431473,2.738334,NaN,NaN,NaN,NaN,NaN
26,Quantile Gradient Boosting conformal,external_test,0.390044,0.434858,0.672045,0.127444,0.258693,0.994348,0.075181,0.434858,NaN,0.995883,6.798735,0.193421,0.148623,0.095883
11,ElasticNet,external_test,0.050943,0.503240,0.770232,0.143372,0.289163,0.992576,0.086165,0.503240,10.722554,NaN,NaN,NaN,NaN,NaN
8,Ridge,external_test,0.039689,0.505969,0.770117,0.143934,0.289933,0.992578,0.086152,0.505969,0.999849,NaN,NaN,NaN,NaN,NaN
2,Persistence forecast,external_test,0.000170,0.806778,1.243489,0.205196,0.483557,0.980649,0.139108,0.806778,NaN,NaN,NaN,NaN,NaN,NaN
5,Seasonal persistence forecast,external_test,0.000201,1.773422,2.727418,0.366644,1.089625,0.906906,0.305113,1.773422,NaN,NaN,NaN,NaN,NaN,NaN
16,ExtraTreesRegressor,test,0.428226,0.402188,0.616060,0.131500,0.289215,0.995435,0.067567,0.402188,31.145975,NaN,NaN,NaN,NaN,NaN


Saved baseline predictions and forecasting metrics.


## Baseline Figures

The figures compare classification, forecasting, error distributions, and calibration behavior. Proposed-model coloring is intentionally not used in baseline-only plots.

In [7]:

# Compact artifact index for baseline computations. Final manuscript figures are generated in Notebook 06.
artifact_rows = []
for path in [
    TABLES / "baseline_classification_metrics.csv",
    TABLES / "baseline_forecasting_metrics.csv",
    TABLES / "baseline_validation_predictions.csv",
    TABLES / "baseline_test_predictions.csv",
    TABLES / "baseline_external_test_predictions.csv",
]:
    if path.exists():
        artifact_rows.append({"artifact": str(path.relative_to(PROJECT_ROOT)), "size_mb": path.stat().st_size / (1024 ** 2)})
artifact_index = pd.DataFrame(artifact_rows)
display(artifact_index)
print("Baseline models trained and saved. Final publication-ready visualizations are centralized in Notebook 06.")


,artifact,size_mb
0,Tables/baseline_classification_metrics.csv,0.010142
1,Tables/baseline_forecasting_metrics.csv,0.005747
2,Tables/baseline_validation_predictions.csv,113.705242
3,Tables/baseline_test_predictions.csv,106.754919
4,Tables/baseline_external_test_predictions.csv,117.103734


Baseline models trained and saved. Final publication-ready visualizations are centralized in Notebook 06.
